# S&P 500 Diebold--Mariano tests

This notebook applies the same design as the Bitcoin DM analysis to the
representative Hybrid ERNN SI-Self and Hybrid QRNN SI-Self specifications.

The date-level loss is the average NLL across seven levels. Seven planned
pairwise comparisons are tested using a Newey--West HAC variance estimate.
Negative DM statistics favour the first named method. Raw two-sided p-values
are primary, and Holm-adjusted p-values are included as a robustness check.
The penalty forecasts come from the final 800-trial runs. No model is retrained.

In [8]:
# ============================================================
# SPY DM TESTS — COMMON SETUP
# Representative specification: Hybrid SI self-lagged
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

PROJECT_DIR = Path(
    "/Users/miaomiaochen/Desktop/PhD program/Project 1"
)
OUTPUT_DIR = PROJECT_DIR / "spy_dm_test_results_800trials"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEVELS = np.array([
    0.025, 0.050, 0.250, 0.500,
    0.750, 0.950, 0.975
], dtype=float)
CODES = [
    "0.025", "0.050", "0.250", "0.500",
    "0.750", "0.950", "0.975"
]

COMPARISONS = [
    ("Method 1b", "Method 3"),
    ("Method 1b", "Sorting"),
    ("Method 1b", "PAVA"),
    ("Method 1b", "Penalty"),
    ("Method 3", "Sorting"),
    ("Method 3", "PAVA"),
    ("Method 3", "Penalty"),
]


def require_files(paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required files:\n" + "\n".join(missing)
        )


def matrix(frame, columns, label):
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(f"{label} is missing columns: {missing}")
    values = frame[columns].to_numpy(dtype=float)
    if values.shape[1] != len(LEVELS):
        raise ValueError(f"Unexpected shape for {label}: {values.shape}")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite forecasts.")
    return values


def validate_actual(reference, candidate, label):
    candidate = np.asarray(candidate, dtype=float)
    if candidate.shape != reference.shape:
        raise ValueError(f"Actual-return length mismatch for {label}.")
    # CSV exports can differ at the last few floating-point digits.
    if not np.allclose(reference, candidate, rtol=0.0, atol=1e-8):
        largest = np.max(np.abs(reference - candidate))
        raise ValueError(
            f"Actual returns do not align for {label}; "
            f"maximum difference={largest:.3e}."
        )


def validate_dates(reference, candidate, label):
    candidate = pd.to_datetime(candidate).to_numpy()
    if not np.array_equal(reference, candidate):
        raise ValueError(f"Test dates do not align for {label}.")


def ordered_sigma(path, level_column):
    frame = pd.read_csv(path)
    required = {level_column, "sigma"}
    if not required.issubset(frame.columns):
        raise KeyError(
            f"{path.name} must contain {sorted(required)}; "
            f"found {list(frame.columns)}"
        )

    saved_levels = frame[level_column].to_numpy(dtype=float)
    saved_sigma = frame["sigma"].to_numpy(dtype=float)
    result = []

    for level in LEVELS:
        match = np.flatnonzero(
            np.isclose(saved_levels, level, atol=1e-10)
        )
        if len(match) != 1:
            raise ValueError(
                f"Expected one sigma for level {level:.3f} in {path}."
            )
        result.append(saved_sigma[match[0]])

    result = np.asarray(result, dtype=float)
    if not np.isfinite(result).all() or np.any(result <= 0):
        raise ValueError(f"Invalid sigma values in {path}.")
    return result


def saved_nll(path, column, reference_dates):
    frame = pd.read_csv(path, parse_dates=["Date"])
    validate_dates(reference_dates, frame["Date"], path.name)
    if column not in frame.columns:
        raise KeyError(f"{path.name} does not contain {column}.")
    values = frame[column].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f"Non-finite NLL values in {path}.")
    return values


def agd_nll(actual, forecasts, sigma):
    residual = actual[:, None] - forecasts
    weight = np.abs(
        LEVELS[None, :] - (residual < 0).astype(float)
    )
    constant = (
        np.log(
            np.sqrt(np.pi / LEVELS)
            + np.sqrt(np.pi / (1.0 - LEVELS))
        )
        - np.log(2.0)
    )
    loss = (
        np.log(sigma)[None, :]
        + constant[None, :]
        + weight * residual**2 / sigma[None, :]**2
    )
    return loss, loss.mean(axis=1)


def ald_nll(actual, forecasts, sigma):
    residual = actual[:, None] - forecasts
    pinball = residual * (
        LEVELS[None, :] - (residual < 0).astype(float)
    )
    loss = (
        -np.log(LEVELS * (1.0 - LEVELS))[None, :]
        + np.log(sigma)[None, :]
        + pinball / sigma[None, :]
    )
    return loss, loss.mean(axis=1)


def newey_west_lrv(series, max_lag):
    centred = np.asarray(series, dtype=float)
    centred = centred - centred.mean()
    sample_size = len(centred)
    result = np.dot(centred, centred) / sample_size

    for lag in range(1, max_lag + 1):
        autocovariance = (
            np.dot(centred[lag:], centred[:-lag])
            / sample_size
        )
        weight = 1.0 - lag / (max_lag + 1.0)
        result += 2.0 * weight * autocovariance
    return result


def dm_test(loss_1, loss_2, max_lag=None):
    loss_1 = np.asarray(loss_1, dtype=float)
    loss_2 = np.asarray(loss_2, dtype=float)

    if loss_1.ndim != 1 or loss_2.ndim != 1:
        raise ValueError("DM loss inputs must be one-dimensional.")
    if loss_1.shape != loss_2.shape:
        raise ValueError("DM loss series must have equal length.")
    if not np.isfinite(loss_1).all() or not np.isfinite(loss_2).all():
        raise ValueError("DM loss series contain non-finite values.")

    difference = loss_1 - loss_2
    sample_size = len(difference)

    if max_lag is None:
        max_lag = int(np.floor(
            4.0 * (sample_size / 100.0) ** (2.0 / 9.0)
        ))

    long_run_variance = newey_west_lrv(
        difference, max_lag
    )
    if (
        not np.isfinite(long_run_variance)
        or long_run_variance <= 0
    ):
        raise ValueError(
            "The DM long-run variance is not positive."
        )

    mean_difference = difference.mean()
    standard_error = np.sqrt(
        long_run_variance / sample_size
    )
    statistic = mean_difference / standard_error
    p_value = 2.0 * norm.sf(abs(statistic))

    return {
        "Mean_loss_difference": mean_difference,
        "DM_statistic": statistic,
        "Raw_p_value": p_value,
        "HAC_lag": max_lag,
        "Long_run_variance": long_run_variance,
    }


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    count = len(p_values)
    order = np.argsort(p_values)
    ordered = p_values[order]
    adjusted_ordered = np.empty(count)
    running_maximum = 0.0

    for rank, p_value in enumerate(ordered):
        adjusted = min(
            1.0,
            max(
                running_maximum,
                (count - rank) * p_value
            )
        )
        adjusted_ordered[rank] = adjusted
        running_maximum = adjusted

    adjusted = np.empty(count)
    adjusted[order] = adjusted_ordered
    return adjusted


def total_crossovers(forecasts, tolerance=1e-12):
    return int(np.sum(
        forecasts[:, :-1]
        > forecasts[:, 1:] + tolerance
    ))


def paths_and_columns(family):
    if family == "ernn":
        return {
            "loss_name": "AGD",
            "loss_function": agd_nll,
            "sort_dir": "spy_ernn_si_self",
            "sort_file": "sorting_spy_ernn_si_self",
            "pava_file": "pava_spy_ernn_si_self",
            "penalty_dir": "spy_ernn_si_self",
            "penalty_file": "penalty_spy_ernn_si_self",
            "method1b_dir": "spy_method1b_joint_ernn_si_self",
            "method1b_file": "spy_method1b_joint_ernn_si_self",
            "method3_root": "spy_method3_spacing_ernn_si_self_results",
            "method3_dir": "ernn_si_self",
            "method3_file": "spy_method3_spacing_ernn_si_self",
            "sorting_columns": [f"sorted_e_{c}" for c in CODES],
            "pava_columns": [f"pava_e_{c}" for c in CODES],
            "method1b_columns": [f"E_{c}" for c in CODES],
            "method3_columns": [f"e_{c}" for c in CODES],
            "penalty_level_column": "tau",
            "method3_level_column": "tau",
            "ranking_nll_column": "Average_AGD_NLL",
        }
    if family == "qrnn":
        return {
            "loss_name": "ALD",
            "loss_function": ald_nll,
            "sort_dir": "spy_qrnn_si_self",
            "sort_file": "sorting_spy_qrnn_si_self",
            "pava_file": "pava_spy_qrnn_si_self",
            "penalty_dir": "spy_qrnn_si_self",
            "penalty_file": "penalty_spy_qrnn_si_self",
            "method1b_dir": "spy_method1b_joint_qrnn_si_self",
            "method1b_file": "spy_method1b_joint_qrnn_si_self",
            "method3_root": "spy_method3_spacing_qrnn_si_self_results",
            "method3_dir": "qrnn_si_self",
            "method3_file": "spy_method3_spacing_qrnn_si_self",
            "sorting_columns": [f"sorted_q_{c}" for c in CODES],
            "pava_columns": [f"pava_q_{c}" for c in CODES],
            "method1b_columns": [f"Q_{c}" for c in CODES],
            "method3_columns": [f"q_{c}" for c in CODES],
            "penalty_level_column": "alpha",
            "method3_level_column": "alpha",
            "ranking_nll_column": "Average_ALD_NLL",
        }
    raise ValueError("family must be 'ernn' or 'qrnn'.")


def run_spy_dm(family):
    cfg = paths_and_columns(family)
    label = family.upper()

    sorting_dir = (
        PROJECT_DIR / "spy_sorting_crossover_results"
        / cfg["sort_dir"]
    )
    pava_dir = (
        PROJECT_DIR / "spy_pava_crossover_results"
        / cfg["sort_dir"]
    )
    penalty_dir = (
        PROJECT_DIR / "spy_penalty_crossover_results_800trials"
        / cfg["penalty_dir"]
    )
    method1b_dir = (
        PROJECT_DIR / "spy_method1b_results"
        / cfg["method1b_dir"]
    )
    method3_dir = (
        PROJECT_DIR / cfg["method3_root"]
        / cfg["method3_dir"]
    )

    files = {
        "sorting_predictions":
            sorting_dir / f'{cfg["sort_file"]}_predictions.csv',
        "sorting_nll":
            sorting_dir / f'{cfg["sort_file"]}_nll_by_date.csv',
        "pava_predictions":
            pava_dir / f'{cfg["pava_file"]}_predictions.csv',
        "pava_nll":
            pava_dir / f'{cfg["pava_file"]}_nll_by_date.csv',
        "penalty_predictions":
            penalty_dir / f'{cfg["penalty_file"]}_test_predictions.csv',
        "penalty_metrics":
            penalty_dir / f'{cfg["penalty_file"]}_test_metrics.csv',
        "method1b_predictions":
            method1b_dir / f'{cfg["method1b_file"]}_test_predictions.csv',
        "method1b_metrics":
            method1b_dir / f'{cfg["method1b_file"]}_test_metrics.csv',
        "method3_predictions":
            method3_dir / f'{cfg["method3_file"]}_test_predictions.csv',
        "method3_metrics":
            method3_dir / f'{cfg["method3_file"]}_test_metrics.csv',
    }
    require_files(files.values())

    sorting = pd.read_csv(
        files["sorting_predictions"],
        parse_dates=["Date"]
    )
    pava = pd.read_csv(
        files["pava_predictions"],
        parse_dates=["Date"]
    )
    penalty = pd.read_csv(files["penalty_predictions"])
    method1b = pd.read_csv(
        files["method1b_predictions"],
        parse_dates=["Date"]
    )
    method3 = pd.read_csv(
        files["method3_predictions"],
        parse_dates=["date"]
    )

    dates = sorting["Date"].to_numpy()
    actual = sorting["Actual"].to_numpy(dtype=float)

    validate_dates(dates, pava["Date"], "PAVA")
    validate_dates(dates, method1b["Date"], "Method 1b")
    validate_dates(dates, method3["date"], "Method 3")
    validate_actual(actual, pava["Actual"], "PAVA")
    validate_actual(actual, penalty["actual"], "Penalty")
    validate_actual(actual, method1b["Actual"], "Method 1b")
    validate_actual(actual, method3["actual"], "Method 3")

    forecasts = {
        "Sorting": matrix(
            sorting, cfg["sorting_columns"], "Sorting"
        ),
        "PAVA": matrix(
            pava, cfg["pava_columns"], "PAVA"
        ),
        "Penalty": matrix(
            penalty, [f"q_{c}" for c in CODES], "Penalty"
        ),
        "Method 1b": matrix(
            method1b, cfg["method1b_columns"], "Method 1b"
        ),
        "Method 3": matrix(
            method3, cfg["method3_columns"], "Method 3"
        ),
    }

    loss_series = {
        "Sorting": saved_nll(
            files["sorting_nll"],
            "sorted_average_nll",
            dates
        ),
        "PAVA": saved_nll(
            files["pava_nll"],
            "pava_average_nll",
            dates
        ),
    }

    sigmas = {
        "Penalty": ordered_sigma(
            files["penalty_metrics"],
            cfg["penalty_level_column"]
        ),
        "Method 1b": ordered_sigma(
            files["method1b_metrics"], "level"
        ),
        "Method 3": ordered_sigma(
            files["method3_metrics"],
            cfg["method3_level_column"]
        ),
    }

    per_level_loss = {}
    for method in ["Penalty", "Method 1b", "Method 3"]:
        (
            per_level_loss[method],
            loss_series[method]
        ) = cfg["loss_function"](
            actual,
            forecasts[method],
            sigmas[method]
        )

    rows = []
    for method_1, method_2 in COMPARISONS:
        result = dm_test(
            loss_series[method_1],
            loss_series[method_2]
        )
        mean_1 = float(np.mean(loss_series[method_1]))
        mean_2 = float(np.mean(loss_series[method_2]))
        rows.append({
            "Comparison": f"{method_1} vs {method_2}",
            "Method_1": method_1,
            "Method_2": method_2,
            "Mean_NLL_1": mean_1,
            "Mean_NLL_2": mean_2,
            **result,
            "Lower_NLL_method":
                method_1 if mean_1 < mean_2 else method_2,
        })

    results = pd.DataFrame(rows)
    results["Holm_p_value"] = holm_adjust(
        results["Raw_p_value"]
    )
    results["Raw_significant_5pct"] = (
        results["Raw_p_value"] < 0.05
    )
    results["Holm_significant_5pct"] = (
        results["Holm_p_value"] < 0.05
    )

    ranking = pd.DataFrame({
        "Method": list(loss_series),
        cfg["ranking_nll_column"]: [
            float(np.mean(loss_series[m]))
            for m in loss_series
        ],
        "Total_Crossover": [
            total_crossovers(forecasts[m])
            for m in loss_series
        ],
    }).sort_values(
        cfg["ranking_nll_column"],
        ascending=True
    ).reset_index(drop=True)
    ranking.insert(
        0, "Rank",
        np.arange(1, len(ranking) + 1)
    )

    output = OUTPUT_DIR / f"{family}_si_self"
    output.mkdir(parents=True, exist_ok=True)

    date_losses = pd.DataFrame({
        "Date": pd.to_datetime(dates)
    })
    for method, values in loss_series.items():
        date_losses[method] = values

    results.to_csv(
        output / f"spy_{family}_si_self_dm_tests.csv",
        index=False
    )
    ranking.to_csv(
        output / f"spy_{family}_si_self_nll_ranking.csv",
        index=False
    )
    date_losses.to_csv(
        output / f"spy_{family}_si_self_nll_by_date.csv",
        index=False
    )

    print("=" * 78)
    print(
        f"SPY HYBRID {label} SI-SELF — "
        f'{cfg["loss_name"]}-NLL DM TESTS'
    )
    print("Negative DM statistics favour the first method.")
    print("Penalty source: final 800-trial run.")
    print("=" * 78)

    display_columns = [
        "Comparison", "Mean_NLL_1", "Mean_NLL_2",
        "DM_statistic", "Raw_p_value",
        "Holm_p_value", "Lower_NLL_method"
    ]
    display(
        results[display_columns].style.format({
            "Mean_NLL_1": "{:.6f}",
            "Mean_NLL_2": "{:.6f}",
            "DM_statistic": "{:.4f}",
            "Raw_p_value": "{:.6f}",
            "Holm_p_value": "{:.6f}",
        })
    )

    print(
        f"\n{label} NLL ranking and "
        "total crossover count:"
    )
    display(
        ranking.style.format({
            cfg["ranking_nll_column"]: "{:.6f}"
        })
    )
    print("\nSaved under:", output)

    return {
        "results": results,
        "ranking": ranking,
        "loss_by_date": date_losses,
        "forecasts": forecasts,
        "sigmas": sigmas,
    }


print("SPY DM-test setup is ready.")
print("Output directory:", OUTPUT_DIR)

SPY DM-test setup is ready.
Output directory: /Users/miaomiaochen/Desktop/PhD program/Project 1/spy_dm_test_results_800trials


In [12]:
# ============================================================
# SPY HYBRID ERNN SI-SELF — AGD-NLL DM TESTS
# ============================================================

spy_ernn_dm = run_spy_dm("ernn")

SPY HYBRID ERNN SI-SELF — AGD-NLL DM TESTS
Negative DM statistics favour the first method.
Penalty source: final 800-trial run.


,Comparison,Mean_NLL_1,Mean_NLL_2,DM_statistic,Raw_p_value,Holm_p_value,Lower_NLL_method
0,Method 1b vs Method 3,-4.100297,-4.103054,0.0861,0.931424,0.973731,Method 3
1,Method 1b vs Sorting,-4.100297,-4.013555,-1.3443,0.178837,0.973731,Method 1b
2,Method 1b vs PAVA,-4.100297,-4.016839,-1.2256,0.220361,0.973731,Method 1b
3,Method 1b vs Penalty,-4.100297,-3.082422,-1.4712,0.141236,0.973731,Method 1b
4,Method 3 vs Sorting,-4.103054,-4.013555,-1.0535,0.292121,0.973731,Method 3
5,Method 3 vs PAVA,-4.103054,-4.016839,-0.9749,0.329587,0.973731,Method 3
6,Method 3 vs Penalty,-4.103054,-3.082422,-1.4791,0.139104,0.973731,Method 3



ERNN NLL ranking and total crossover count:


,Rank,Method,Average_AGD_NLL,Total_Crossover
0,1,Method 3,-4.103054,0
1,2,Method 1b,-4.100297,0
2,3,PAVA,-4.016839,0
3,4,Sorting,-4.013555,0
4,5,Penalty,-3.082422,3



Saved under: /Users/miaomiaochen/Desktop/PhD program/Project 1/spy_dm_test_results_800trials/ernn_si_self


In [10]:
# ============================================================
# SPY HYBRID QRNN SI-SELF — ALD-NLL DM TESTS
# ============================================================

spy_qrnn_dm = run_spy_dm("qrnn")

SPY HYBRID QRNN SI-SELF — ALD-NLL DM TESTS
Negative DM statistics favour the first method.
Penalty source: final 800-trial run.


,Comparison,Mean_NLL_1,Mean_NLL_2,DM_statistic,Raw_p_value,Holm_p_value,Lower_NLL_method
0,Method 1b vs Method 3,-4.001669,-4.059417,3.0864,0.002026,0.008104,Method 3
1,Method 1b vs Sorting,-4.001669,-3.917714,-3.0727,0.002121,0.008104,Method 1b
2,Method 1b vs PAVA,-4.001669,-3.909544,-3.1829,0.001458,0.007292,Method 1b
3,Method 1b vs Penalty,-4.001669,-3.903694,-0.9883,0.322994,0.322994,Method 1b
4,Method 3 vs Sorting,-4.059417,-3.917714,-4.7029,0.000003,0.000018,Method 3
5,Method 3 vs PAVA,-4.059417,-3.909544,-4.6878,0.000003,0.000018,Method 3
6,Method 3 vs Penalty,-4.059417,-3.903694,-1.5207,0.128347,0.256694,Method 3



QRNN NLL ranking and total crossover count:


,Rank,Method,Average_ALD_NLL,Total_Crossover
0,1,Method 3,-4.059417,0
1,2,Method 1b,-4.001669,0
2,3,Sorting,-3.917714,0
3,4,PAVA,-3.909544,0
4,5,Penalty,-3.903694,7



Saved under: /Users/miaomiaochen/Desktop/PhD program/Project 1/spy_dm_test_results_800trials/qrnn_si_self
